In [ ]:
!pip install openai==0.28

In [ ]:
!pip install faiss-cpu

In [ ]:
!pip install rank_bm25

In [ ]:
!pip install pandas

# Importing Dependencies

In [3]:
import openai
import pandas as pd
import numpy as np
import re
import time
import random

In [4]:
import json

In [ ]:
openai.api_key = "API KEY"

# Datasets

In [6]:
df = pd.read_csv("../Code_Snippets.csv")

In [7]:
df.head()

,ID,Problem Title,Problem Description,Submitted Code,Problem Type,Source
0,1,Find All K-Distant Indices in an Array,You are given a 0-indexed integer array nums a...,class Solution:\r\n def findKDistantIndices...,Easy,LeetCode
1,2,Two Sum,Given an array of integers nums and an integer...,"class Solution:\r\n def twoSum(self, nums: ...",Easy,LeetCode
2,3,Symmetric Tree,"Given the root of a binary tree, check whether...","class Solution:\r\n def isSymmetric(self, r...",Easy,LeetCode
3,4,Same Tree,"Given the roots of two binary trees p and q, w...","class Solution:\r\n def isSameTree(self, p:...",Easy,LeetCode
4,5,Binary Tree Inorder Traversal,"Given the root of a binary tree, return the in...",class Solution:\r\n def inorderTraversal(se...,Easy,LeetCode


In [8]:
df.shape

(300, 6)

In [9]:
human_eval_df = pd.read_csv("../rag_data/test.csv")

In [10]:
human_eval_df.shape

(164, 5)

In [11]:
file_path = '../rag_data/mbpp.jsonl'
data = []
with open(file_path, 'r') as file:
    for line in file:
        data.append(json.loads(line))

mbpp_df = pd.DataFrame(data)
mbpp_df.shape

(974, 6)

In [12]:
task_descriptions = (df['Problem Title'] + " " + df['Problem Description']).tolist()
code_snippets = df['Submitted Code'].tolist()

# FewShot Prompt 2

In [13]:
MAX_RETRIES = 7
BASE_DELAY = 7
start_index = 0
data = []

In [ ]:
for i, (code_snippet, task_description) in enumerate(zip(code_snippets[start_index:], task_descriptions[start_index:]), start=start_index + 1):
  retries = 0
  success = False

  while retries < MAX_RETRIES:
    try:
      # Generate the response for the code snippet using chat-completions endpoint
      response = openai.ChatCompletion.create(
          model="gpt-4o",
          messages=[
              {
                  "role": "system",
                  "content": """You are a helpful AI assistant working with developers to create unit test cases from codes to help the development process."""
              },
              {
                  "role": "user",
                  "content": """You are given a Python function along with its task description. Your goal is to generate
                    a complete and executable set of test cases for the function. The test cases should be written using Python
                    using the unittest framework and should comprehensively cover various scenarios, including:
                    * Typical cases (common expected inputs).
                    * Edge cases (unusual but valid inputs).
                    * Boundary conditions (extreme values or limits).
                    The output should be a fully standalone Python script that can be executed without modification. Ensure that the
                    generated test script includes:
                    (1) A properly structured TestSolution class with multiple test_ methods.
                    (2) Descriptive test method names that clearly indicate the scenario being tested.
                    (3) Assertions validating the correctness of the function's output.
                    (4) The function should be encapsulated within a Solution class for better organization.

                    Important Requirements:
                    * Include the if __name__ == '__main__': unittest.main() block to allow direct execution.
                    * Do not provide any explanations—only return the complete test script.

                    Here are some examples:

                        Example 1:
                        -------------------------
                        Task Description: Write a function that returns the factorial of a given number.

                        Code Snippet:
                        def factorial(n):
                            if n == 0 or n == 1:
                                return 1
                            return n * factorial(n - 1)
    
                        Output:
                        ```python
                        import unittest

                        class Solution:
                            def factorial(self, n):
                                if n == 0 or n == 1:
                                    return 1
                                return n * self.factorial(n - 1)
                        
                        class TestSolution(unittest.TestCase):
                            def setUp(self):
                                self.solution = Solution()
                        
                            def test_factorial_of_0(self):
                                self.assertEqual(self.solution.factorial(0), 1)
                        
                            def test_factorial_of_1(self):
                                self.assertEqual(self.solution.factorial(1), 1)
                        
                            def test_factorial_of_3(self):
                                self.assertEqual(self.solution.factorial(3), 6)
                        
                            def test_factorial_of_5(self):
                                self.assertEqual(self.solution.factorial(5), 120)
                        
                            def test_factorial_of_7(self):
                                self.assertEqual(self.solution.factorial(7), 5040)
                        
                            def test_factorial_of_10(self):
                                self.assertEqual(self.solution.factorial(10), 3628800)
                        
                            def test_factorial_of_12(self):
                                self.assertEqual(self.solution.factorial(12), 479001600)
                        
                        if __name__ == '__main__':
                            unittest.main()
                        ```

                        Example 2:
                    -------------------------
                    Task Description: Write a function that reverses a string.

                    Code Snippet:
                    def reverse_string(s):
                        return s[::-1]

                    Output:
                    ```python
                    import unittest

                    class Solution:
                        def reverse_string(self, s):
                            return s[::-1]
                    
                    class TestSolution(unittest.TestCase):
                        def setUp(self):
                            self.solution = Solution()
                    
                        def test_reverse_string_hello(self):
                            self.assertEqual(self.solution.reverse_string("hello"), "olleh")
                    
                        def test_reverse_string_world(self):
                            self.assertEqual(self.solution.reverse_string("world"), "dlrow")
                    
                        def test_reverse_string_empty(self):
                            self.assertEqual(self.solution.reverse_string(""), "")
                    
                        def test_reverse_string_single_char(self):
                            self.assertEqual(self.solution.reverse_string("a"), "a")
                    
                        def test_reverse_string_special_characters(self):
                            self.assertEqual(self.solution.reverse_string("123@abc"), "cba@321")
                    
                        def test_reverse_string_mixed_case(self):
                            self.assertEqual(self.solution.reverse_string("AbCDeF"), "FeDCbA")
                    
                        def test_reverse_string_with_spaces(self):
                            self.assertEqual(self.solution.reverse_string("Open AI"), "IA nepO")
                    
                    if __name__ == '__main__':
                        unittest.main()
                    ```

                    Now generate the test cases for the following function:

                    Now generate the test cases for the following function:

                    Task Description: {task_description}

                    Code Snippet:
                    {code_snippet}

                    Output:
                    Test Script: [generate ONLY the test script]
                    """
              }
          ],
          temperature=0.7
      )
      data.append({'Code Snippet': code_snippet, 'Response': response['choices'][0]['message']['content'].strip()})
      print(f"{i}. Processed successfully")
      print(response['choices'][0]['message']['content'].strip())
      time.sleep(BASE_DELAY)  # Apply base delay after success
      success = True
      break

    except Exception as e:
      print(f"Error processing snippet {i}, attempt {retries + 1}: {e}")
      retries += 1
      if "429" in str(e):  # Rate limit error
        delay = BASE_DELAY * (2 ** retries) + random.uniform(1, 3)
        print(f"Rate limit hit. Retrying in {delay:.2f} seconds...")
        time.sleep(delay)
      else:
        break  # Break for non-rate-limit errors

    if not success:
      # Append fallback response if all retries fail
      data.append({'Code Snippet': code_snippet, 'Response': "Cannot generate"})
      print(f"Failed to process snippet {i}. Defaulted to 'Cannot generate'.")

In [ ]:
output_df = pd.DataFrame(data)
output_df.to_csv("Fewshot_Prompt2.csv", index=False)
print("Test cases saved to 'Fewshot_Prompt2.csv'")

# RAG (FAISS)

In [ ]:
!pip install tiktoken

In [ ]:
!pip install transformers

In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install ipywidgets
!pip install --upgrade notebook

In [ ]:
!pip install jupyterlab_widgets

In [22]:
import faiss
from sentence_transformers import SentenceTransformer

In [14]:
human_eval_df["prompt"] = human_eval_df["prompt"].astype(str)
human_eval_df["canonical_solution"] = human_eval_df["canonical_solution"].astype(str)
human_eval_df["test"] = human_eval_df["test"].astype(str)

In [15]:
mbpp_df["text"] = mbpp_df["text"].astype(str)
mbpp_df["code"] = mbpp_df["code"].astype(str)
mbpp_df["test_list"] = mbpp_df["test_list"].astype(str)

In [16]:
human_eval_texts = human_eval_df["prompt"] + " " + human_eval_df["canonical_solution"] + " " + human_eval_df["test"]
mbpp_texts = mbpp_df["text"] + " " + mbpp_df["code"] + " " + mbpp_df["test_list"]

In [17]:
all_texts = list(human_eval_texts) + list(mbpp_texts)

In [23]:
# Load embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
embeddings = model.encode(all_texts, convert_to_numpy=True)

In [ ]:
# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Save index
faiss.write_index(index, "faiss_index.bin")

# Save mapping of text to index
df_mapping = pd.DataFrame({"text": all_texts})
df_mapping.to_csv("index_mapping.csv", index=False)

In [ ]:
# Function to retrieve relevant examples
def retrieve_examples(query, top_k=3):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    results = [df_mapping.iloc[i]["text"] for i in indices[0]]
    return results

In [ ]:
data2 = []

In [ ]:
for i, (code_snippet, task_description) in enumerate(zip(code_snippets[start_index:], task_descriptions[start_index:]), start=start_index + 1):
    retries = 0
    success = False

    # Retrieve similar examples using FAISS
    retrieved_examples = retrieve_examples(task_description, top_k=3)
    retrieved_text = "\n\n".join(retrieved_examples)

    while retries < MAX_RETRIES:
        try:
            # Generate the response for the code snippet using chat-completions endpoint
            response = openai.ChatCompletion.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "system",
                        "content": f"""You are a helpful AI assistant working with developers to create unit test cases
                        from codes to help the development process. You have access to a retrieval system that provides
                        relevant past examples to assist in generating high-quality test cases."""
                    },
                    {
                        "role": "user",
                        "content": f"""You are given a Python function along with its task description. Your goal is to generate
                    a complete and executable set of test cases for the function. The test cases should be written using Python
                    using the unittest framework and should comprehensively cover various scenarios, including:
                    * Typical cases (common expected inputs).
                    * Edge cases (unusual but valid inputs).
                    * Boundary conditions (extreme values or limits).
                    The output should be a fully standalone Python script that can be executed without modification. Ensure that the
                    generated test script includes:
                    (1) A properly structured TestSolution class with multiple test_ methods.
                    (2) Descriptive test method names that clearly indicate the scenario being tested.
                    (3) Assertions validating the correctness of the function's output.
                    (4) The function should be encapsulated within a Solution class for better organization.

                    Important Requirements:
                    * Include the if __name__ == '__main__': unittest.main() block to allow direct execution.
                    * Do not provide any explanations—only return the complete test script.

                    Here are some relevant test case examples retrieved from a similar problem to improve quality:
                        {retrieved_text}

                    Here are some examples:

                        Example 1:
                        -------------------------
                        Task Description: Write a function that returns the factorial of a given number.

                        Code Snippet:
                        def factorial(n):
                            if n == 0 or n == 1:
                                return 1
                            return n * factorial(n - 1)
    
                        Output:
                        ```python
                        import unittest

                        class Solution:
                            def factorial(self, n):
                                if n == 0 or n == 1:
                                    return 1
                                return n * self.factorial(n - 1)
                        
                        class TestSolution(unittest.TestCase):
                            def setUp(self):
                                self.solution = Solution()
                        
                            def test_factorial_of_0(self):
                                self.assertEqual(self.solution.factorial(0), 1)
                        
                            def test_factorial_of_1(self):
                                self.assertEqual(self.solution.factorial(1), 1)
                        
                            def test_factorial_of_3(self):
                                self.assertEqual(self.solution.factorial(3), 6)
                        
                            def test_factorial_of_5(self):
                                self.assertEqual(self.solution.factorial(5), 120)
                        
                            def test_factorial_of_7(self):
                                self.assertEqual(self.solution.factorial(7), 5040)
                        
                            def test_factorial_of_10(self):
                                self.assertEqual(self.solution.factorial(10), 3628800)
                        
                            def test_factorial_of_12(self):
                                self.assertEqual(self.solution.factorial(12), 479001600)
                        
                        if __name__ == '__main__':
                            unittest.main()
                        ```

                        Example 2:
                    -------------------------
                    Task Description: Write a function that reverses a string.

                    Code Snippet:
                    def reverse_string(s):
                        return s[::-1]

                    Output:
                    ```python
                    import unittest

                    class Solution:
                        def reverse_string(self, s):
                            return s[::-1]
                    
                    class TestSolution(unittest.TestCase):
                        def setUp(self):
                            self.solution = Solution()
                    
                        def test_reverse_string_hello(self):
                            self.assertEqual(self.solution.reverse_string("hello"), "olleh")
                    
                        def test_reverse_string_world(self):
                            self.assertEqual(self.solution.reverse_string("world"), "dlrow")
                    
                        def test_reverse_string_empty(self):
                            self.assertEqual(self.solution.reverse_string(""), "")
                    
                        def test_reverse_string_single_char(self):
                            self.assertEqual(self.solution.reverse_string("a"), "a")
                    
                        def test_reverse_string_special_characters(self):
                            self.assertEqual(self.solution.reverse_string("123@abc"), "cba@321")
                    
                        def test_reverse_string_mixed_case(self):
                            self.assertEqual(self.solution.reverse_string("AbCDeF"), "FeDCbA")
                    
                        def test_reverse_string_with_spaces(self):
                            self.assertEqual(self.solution.reverse_string("Open AI"), "IA nepO")
                    
                    if __name__ == '__main__':
                        unittest.main()
                    ```

                    Now generate the test cases for the following function:

                    Task Description: {task_description}

                    Code Snippet:
                    {code_snippet}

                    Output:
                    Test Script: [generate ONLY the test script]
                    """
                    }
                ],
                temperature=0.0
            )
            data2.append({'Code Snippet': code_snippet, 'Response': response['choices'][0]['message']['content'].strip()})
            print(f"{i}. Processed successfully")
            print(response['choices'][0]['message']['content'].strip())
            time.sleep(BASE_DELAY)  # Apply base delay after success
            success = True
            break

        except Exception as e:
            print(f"Error processing snippet {i}, attempt {retries + 1}: {e}")
            retries += 1
            if "429" in str(e):  # Rate limit error
                delay = BASE_DELAY * (2 ** retries) + random.uniform(1, 3)
                print(f"Rate limit hit. Retrying in {delay:.2f} seconds...")
                time.sleep(delay)
            else:
                break  # Break for non-rate-limit errors

    if not success:
        # Append fallback response if all retries fail
        data2.append({'Code Snippet': code_snippet, 'Response': "Cannot generate"})
        print(f"Failed to process snippet {i}. Defaulted to 'Cannot generate'.")

In [ ]:
output_df2 = pd.DataFrame(data2)
output_df2.to_csv("FAISS_Fewshot_Prompt2.csv", index=False)
print("Test cases saved to 'FAISS_Fewshot_Prompt2.csv'")

# RAG (BM25)

In [ ]:
from rank_bm25 import BM25Okapi

In [ ]:
human_eval_df["prompt"] = human_eval_df["prompt"].astype(str)
human_eval_df["canonical_solution"] = human_eval_df["canonical_solution"].astype(str)
human_eval_df["test"] = human_eval_df["test"].astype(str)

mbpp_df["text"] = mbpp_df["text"].astype(str)
mbpp_df["code"] = mbpp_df["code"].astype(str)
mbpp_df["test_list"] = mbpp_df["test_list"].astype(str)

In [ ]:
all_tasks = human_eval_df['prompt'].tolist() + mbpp_df['text'].tolist()
all_codes = human_eval_df['canonical_solution'].tolist() + mbpp_df['code'].tolist()
all_tests = human_eval_df['test'].tolist() + mbpp_df['test_list'].tolist()

In [ ]:
# Tokenize text for BM25
tokenized_corpus = [task.split() for task in all_tasks]

In [ ]:
# Fit BM25 model
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def retrieve_examples(task_description, top_k=3):
    tokenized_query = task_description.split()
    scores = bm25.get_scores(tokenized_query)

    # Get top-k indices sorted by score
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    # Retrieve corresponding code snippets
    retrieved = [f"Task: {all_tasks[i]}\nCode: {all_codes[i]}\nTest: {all_tests[i]}" for i in top_indices]
    return retrieved

In [ ]:
data3 = []
for i, (code_snippet, task_description) in enumerate(zip(code_snippets[start_index:], task_descriptions[start_index:]), start=start_index + 1):
    retries = 0
    success = False

    # Retrieve examples using BM25
    retrieved_examples = retrieve_examples(task_description, top_k=3)
    retrieved_text = "\n\n".join(retrieved_examples)

    while retries < MAX_RETRIES:
        try:
            # Generate response using OpenAI API
            response = openai.ChatCompletion.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "system",
                        "content": f"""You are a helpful AI assistant working with developers to create unit test cases from codes
                        to help the development process. You have access to a retrieval system that provides relevant past examples
                        to assist in generating high-quality test cases."""
                    },
                    {
                        "role": "user",
                        "content": f"""You are given a Python function along with its task description. Your goal is to generate
                    a complete and executable set of test cases for the function. The test cases should be written using Python
                    using the unittest framework and should comprehensively cover various scenarios, including:
                    * Typical cases (common expected inputs).
                    * Edge cases (unusual but valid inputs).
                    * Boundary conditions (extreme values or limits).
                    The output should be a fully standalone Python script that can be executed without modification. Ensure that the
                    generated test script includes:
                    (1) A properly structured TestSolution class with multiple test_ methods.
                    (2) Descriptive test method names that clearly indicate the scenario being tested.
                    (3) Assertions validating the correctness of the function's output.
                    (4) The function should be encapsulated within a Solution class for better organization.

                    Important Requirements:
                    * Include the if __name__ == '__main__': unittest.main() block to allow direct execution.
                    * Do not provide any explanations—only return the complete test script.

                    Here are some relevant test case examples retrieved from a similar problem to improve quality:
                        {retrieved_text}

                    Here are some examples:

                        Example 1:
                        -------------------------
                        Task Description: Write a function that returns the factorial of a given number.

                        Code Snippet:
                        def factorial(n):
                            if n == 0 or n == 1:
                                return 1
                            return n * factorial(n - 1)
    
                        Output:
                        ```python
                        import unittest

                        class Solution:
                            def factorial(self, n):
                                if n == 0 or n == 1:
                                    return 1
                                return n * self.factorial(n - 1)
                        
                        class TestSolution(unittest.TestCase):
                            def setUp(self):
                                self.solution = Solution()
                        
                            def test_factorial_of_0(self):
                                self.assertEqual(self.solution.factorial(0), 1)
                        
                            def test_factorial_of_1(self):
                                self.assertEqual(self.solution.factorial(1), 1)
                        
                            def test_factorial_of_3(self):
                                self.assertEqual(self.solution.factorial(3), 6)
                        
                            def test_factorial_of_5(self):
                                self.assertEqual(self.solution.factorial(5), 120)
                        
                            def test_factorial_of_7(self):
                                self.assertEqual(self.solution.factorial(7), 5040)
                        
                            def test_factorial_of_10(self):
                                self.assertEqual(self.solution.factorial(10), 3628800)
                        
                            def test_factorial_of_12(self):
                                self.assertEqual(self.solution.factorial(12), 479001600)
                        
                        if __name__ == '__main__':
                            unittest.main()
                        ```

                        Example 2:
                    -------------------------
                    Task Description: Write a function that reverses a string.

                    Code Snippet:
                    def reverse_string(s):
                        return s[::-1]

                    Output:
                    ```python
                    import unittest

                    class Solution:
                        def reverse_string(self, s):
                            return s[::-1]
                    
                    class TestSolution(unittest.TestCase):
                        def setUp(self):
                            self.solution = Solution()
                    
                        def test_reverse_string_hello(self):
                            self.assertEqual(self.solution.reverse_string("hello"), "olleh")
                    
                        def test_reverse_string_world(self):
                            self.assertEqual(self.solution.reverse_string("world"), "dlrow")
                    
                        def test_reverse_string_empty(self):
                            self.assertEqual(self.solution.reverse_string(""), "")
                    
                        def test_reverse_string_single_char(self):
                            self.assertEqual(self.solution.reverse_string("a"), "a")
                    
                        def test_reverse_string_special_characters(self):
                            self.assertEqual(self.solution.reverse_string("123@abc"), "cba@321")
                    
                        def test_reverse_string_mixed_case(self):
                            self.assertEqual(self.solution.reverse_string("AbCDeF"), "FeDCbA")
                    
                        def test_reverse_string_with_spaces(self):
                            self.assertEqual(self.solution.reverse_string("Open AI"), "IA nepO")
                    
                    if __name__ == '__main__':
                        unittest.main()
                    ```

                    Now generate the test cases for the following function:

                    Task Description: {task_description}

                    Code Snippet:
                    {code_snippet}

                    Output:
                    Test Script: [generate ONLY the test script]
                        """
                    }
                ],
                temperature=0.7
            )
            data3.append({'Code Snippet': code_snippet, 'Response': response['choices'][0]['message']['content'].strip()})
            print(f"{i}. Processed successfully")
            print(response['choices'][0]['message']['content'].strip())
            time.sleep(BASE_DELAY)  # Apply base delay after success
            success = True
            break

        except Exception as e:
            print(f"Error processing snippet {i}, attempt {retries + 1}: {e}")
            retries += 1
            if "429" in str(e):  # Rate limit error
                delay = BASE_DELAY * (2 ** retries) + random.uniform(1, 3)
                print(f"Rate limit hit. Retrying in {delay:.2f} seconds...")
                time.sleep(delay)
            else:
                break  # Break for non-rate-limit errors

    if not success:
        data3.append({'Code Snippet': code_snippet, 'Response': "Cannot generate"})
        print(f"Failed to process snippet {i}. Defaulted to 'Cannot generate'.")

In [ ]:
output_df3 = pd.DataFrame(data3)
output_df3.to_csv("BM25_Fewshot_Prompt2.csv", index=False)
print("Test cases saved to 'BM25_Fewshot_Prompt2.csv'")